# Accuracy test for monthly sales forecasts

This notebook evaluates quantity forecasts; it does not create or overwrite the production forecast. Set the test horizon below (for example 6 or 12 months), then run the cells in order. It compares LightGBM against seasonal naive using historical actuals.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMRegressor

# Update these paths if the project or data files move.
project_dir = Path(r"C:\Divyesh\Technoval Forecasting")
cleaned_sales_file = project_dir / "Cleaned_sales_data.parquet"
monthly_sales_file = project_dir / "Monthly_sales.parquet"

# Choose any positive number of holdout months supported by the available history.
holdout_months = 12  # For example: 6, 12, or another valid number of months.
manual_cutoff_month = None  # Example: "2026-09-01"; set only after confirming the month is complete.

series_keys = ["Company", "Customer", "Product Family", "Manufacturer or Trading"]
attribute_columns = ["Territory", "Customer Type"]

for required_path in [cleaned_sales_file, monthly_sales_file]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required input file not found: {required_path}")

cleaned_sales = pd.read_parquet(cleaned_sales_file)
monthly_sales = pd.read_parquet(monthly_sales_file)
cleaned_sales["InvDate"] = pd.to_datetime(cleaned_sales["InvDate"], errors="raise")
monthly_sales["Month"] = pd.to_datetime(monthly_sales["Month"], errors="raise").dt.to_period("M").dt.to_timestamp()

latest_source_date = cleaned_sales["InvDate"].max().normalize()
latest_source_month = latest_source_date.to_period("M").to_timestamp()
calendar_month_end = latest_source_month + pd.offsets.MonthEnd(0)
if manual_cutoff_month is not None:
    forecast_cutoff_month = pd.Timestamp(manual_cutoff_month).to_period("M").to_timestamp()
else:
    forecast_cutoff_month = (
        latest_source_month
        if latest_source_date == calendar_month_end
        else latest_source_month - pd.offsets.MonthBegin(1)
    )

if forecast_cutoff_month not in set(monthly_sales["Month"]):
    raise ValueError(
        f"Cutoff {forecast_cutoff_month:%Y-%m} is not present in Monthly_sales.parquet. "
        "Rebuild the monthly file or choose a valid cutoff."
    )
analysis_monthly = monthly_sales.loc[monthly_sales["Month"] <= forecast_cutoff_month].copy()
available_months = pd.DatetimeIndex(analysis_monthly["Month"].drop_duplicates().sort_values())
if not isinstance(holdout_months, int) or isinstance(holdout_months, bool) or holdout_months < 1:
    raise ValueError("holdout_months must be a positive integer, such as 6 or 12.")
if holdout_months >= len(available_months):
    raise ValueError(
        f"Requested {holdout_months} holdout months, but only {len(available_months)} months are available; "
        "at least one earlier month is needed for training."
    )

backtest_months = pd.date_range(end=forecast_cutoff_month, periods=holdout_months, freq="MS")
missing_test_months = backtest_months.difference(available_months)
if not missing_test_months.empty:
    raise ValueError(f"Monthly data is missing holdout months: {missing_test_months.strftime('%Y-%m').tolist()}")
backtest_start = backtest_months[0]
analysis_monthly = analysis_monthly.sort_values(series_keys + ["Month"]).reset_index(drop=True)
print(f"Cleaned source: {cleaned_sales_file}")
print(f"Monthly series: {monthly_sales_file}")
print(f"Available data: {available_months.min():%Y-%m} to {available_months.max():%Y-%m}; cutoff: {forecast_cutoff_month:%Y-%m}")
print(f"Holdout length: {holdout_months} months ({backtest_months[0]:%Y-%m} to {backtest_months[-1]:%Y-%m})")
print(f"Monthly rows available for test: {len(analysis_monthly):,}; series keys: {len(series_keys)}")

Cleaned source: C:\Divyesh\Technoval Forecasting\Cleaned_sales_data.parquet
Monthly series: C:\Divyesh\Technoval Forecasting\Monthly_sales.parquet
Available data: 2018-04 to 2026-09; cutoff: 2026-09
Holdout length: 12 months (2025-10 to 2026-09)
Monthly rows available for test: 366,812; series keys: 4


## Features for the test

These feature and recursive-forecast functions match the forecasting notebook. All lag and rolling features use earlier months only. Territory and Customer Type are retained for report rollups, not used as model inputs.

In [2]:
from lightgbm import LGBMRegressor

# Territory and Customer Type remain available for reports, but are excluded from model inputs.
categorical_features = series_keys
numeric_features = (
    [f"{target}_lag_{lag}" for target in ["qty", "net_amt"] for lag in [1, 2, 3, 6, 12]]
    + [f"{target}_roll_mean_{window}" for target in ["qty", "net_amt"] for window in [3, 6, 12]]
    + ["active_months_12", "months_since_last_sale", "month_number", "quarter"]
)
feature_columns = categorical_features + numeric_features
category_dtypes = {
    column: pd.CategoricalDtype(categories=sorted(analysis_monthly[column].dropna().astype(str).unique()))
    for column in categorical_features
}

def add_historical_features(frame):
    result = frame.sort_values(series_keys + ["Month"]).copy()
    grouped = result.groupby(series_keys, observed=True, sort=False)
    for target in ["qty", "net_amt"]:
        for lag in [1, 2, 3, 6, 12]:
            result[f"{target}_lag_{lag}"] = grouped[target].shift(lag)
        for window in [3, 6, 12]:
            result[f"{target}_roll_mean_{window}"] = grouped[target].transform(
                lambda values: values.shift(1).rolling(window, min_periods=1).mean()
            )
    result["month_number"] = result["Month"].dt.month.astype("int8")
    result["quarter"] = result["Month"].dt.quarter.astype("int8")
    result["active_months_12"] = grouped["qty"].transform(
        lambda values: values.shift(1).gt(0).rolling(12, min_periods=1).sum()
    )
    result["_month_index"] = result["Month"].dt.year * 12 + result["Month"].dt.month
    result["_last_sale_month"] = result["_month_index"].where(result["qty"].gt(0))
    result["_last_sale_month"] = result.groupby(series_keys, observed=True, sort=False)["_last_sale_month"].ffill()
    result["_last_sale_month"] = result.groupby(series_keys, observed=True, sort=False)["_last_sale_month"].shift(1)
    result["months_since_last_sale"] = result["_month_index"] - result["_last_sale_month"]
    result["months_since_last_sale"] = result["months_since_last_sale"].fillna(999).astype("float32")
    result = result.drop(columns=["_month_index", "_last_sale_month"])
    for column, dtype in category_dtypes.items():
        result[column] = result[column].astype("string").astype(dtype)
    return result

def recursive_quantity_forecast(history, future_months, model=None):
    states = {}
    ordered_history = history.sort_values(series_keys + ["Month"])
    for key, group in ordered_history.groupby(series_keys, observed=True, sort=False):
        if not isinstance(key, tuple):
            key = (key,)
        group = group.sort_values("Month")
        dates = pd.to_datetime(group["Month"])
        month_indices = (dates.dt.year * 12 + dates.dt.month).to_numpy(dtype="int64")
        qty_values = pd.to_numeric(group["qty"], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
        net_values = pd.to_numeric(group["net_amt"], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
        month_values = {int(month): (float(qty), float(amount)) for month, qty, amount in zip(month_indices, qty_values, net_values)}
        positive_prices = [
            (month, amount / qty)
            for month, qty, amount in zip(month_indices, qty_values, net_values)
            if qty > 0 and np.isfinite(amount / qty) and amount / qty > 0
        ]
        latest = group.iloc[-1]
        attributes = {column: latest[column] for column in series_keys + attribute_columns}
        states[key] = {
            "attributes": attributes,
            "months": month_values,
            "qty_history": qty_values.tolist(),
            "net_history": net_values.tolist(),
            "positive_prices": positive_prices,
            "last_sale_month": max((month for month, qty in zip(month_indices, qty_values) if qty > 0), default=None),
        }

    forecast_steps = []
    for future_month in future_months:
        future_month = pd.Timestamp(future_month)
        future_index = future_month.year * 12 + future_month.month
        feature_records = []
        keys_in_order = list(states)
        if model is not None:
            for key in keys_in_order:
                state = states[key]
                record = dict(state["attributes"])
                for target, values in [("qty", state["qty_history"]), ("net_amt", state["net_history"])]:
                    for lag in [1, 2, 3, 6, 12]:
                        record[f"{target}_lag_{lag}"] = values[-lag] if len(values) >= lag else np.nan
                    for window in [3, 6, 12]:
                        recent_values = values[-window:]
                        record[f"{target}_roll_mean_{window}"] = float(np.mean(recent_values)) if recent_values else np.nan
                record["active_months_12"] = sum(value > 0 for value in state["qty_history"][-12:])
                record["months_since_last_sale"] = future_index - state["last_sale_month"] if state["last_sale_month"] is not None else 999
                record["month_number"] = future_month.month
                record["quarter"] = future_month.quarter
                feature_records.append(record)
            current_features = pd.DataFrame(feature_records)
            for column, dtype in category_dtypes.items():
                current_features[column] = current_features[column].astype("string").astype(dtype)
            qty_predictions = np.maximum(model.predict(current_features[feature_columns]), 0)
        else:
            qty_predictions = []
            for key in keys_in_order:
                months = states[key]["months"]
                seasonal = months.get(future_index - 12)
                latest = months[max(months)] if months else None
                qty_predictions.append(max((seasonal or latest or (0, 0))[0], 0))
            qty_predictions = np.asarray(qty_predictions, dtype="float64")

        family_recent_prices = {}
        all_recent_prices = []
        for state in states.values():
            family = state["attributes"]["Product Family"]
            for month_index, (qty, amount) in state["months"].items():
                if future_index - 12 <= month_index < future_index and qty > 0:
                    price = amount / qty
                    if np.isfinite(price) and price > 0:
                        family_recent_prices.setdefault(family, []).append(price)
                        all_recent_prices.append(price)
        family_medians = {family: float(np.median(prices)) for family, prices in family_recent_prices.items()}
        global_median = float(np.median(all_recent_prices)) if all_recent_prices else np.nan

        step_rows = []
        for key, qty_prediction in zip(keys_in_order, qty_predictions):
            state = states[key]
            recent_series_prices = [price for _, price in state["positive_prices"][-3:]]
            if recent_series_prices:
                price_estimate = float(np.mean(recent_series_prices))
                price_source = "series_recent_3"
            else:
                family = state["attributes"]["Product Family"]
                price_estimate = family_medians.get(family, np.nan)
                price_source = "product_family_recent_median" if np.isfinite(price_estimate) else "no_historical_price"
                if not np.isfinite(price_estimate) and np.isfinite(global_median):
                    price_estimate = global_median
                    price_source = "global_recent_median"
            if qty_prediction > 0 and not np.isfinite(price_estimate):
                raise ValueError("Positive forecast quantity has no usable historical price; resolve price coverage before export.")
            net_prediction = qty_prediction * price_estimate if np.isfinite(price_estimate) else 0.0
            row = dict(state["attributes"])
            row.update({
                "Month": future_month,
                "qty": float(qty_prediction),
                "net_amt": float(net_prediction),
                "price_estimate": price_estimate,
                "price_source": price_source,
            })
            step_rows.append(row)
            state["months"][future_index] = (float(qty_prediction), float(net_prediction))
            state["qty_history"].append(float(qty_prediction))
            state["net_history"].append(float(net_prediction))
            if qty_prediction > 0 and np.isfinite(price_estimate) and price_estimate > 0:
                state["positive_prices"].append((future_index, float(price_estimate)))
                state["last_sale_month"] = future_index
        forecast_steps.append(pd.DataFrame(step_rows))
    return pd.concat(forecast_steps, ignore_index=True)

## Run the holdout and compare models

The model is trained only on months before the selected holdout. During the holdout, each predicted month becomes history for the next predicted month. Metrics compare predicted quantity with actual quantity; new series that first appear during the test are reported but excluded from model scoring.

In [3]:
# Build leakage-safe features, train before the holdout, and compare recursive forecasts.
all_features = add_historical_features(analysis_monthly)
train_rows = all_features.loc[all_features["Month"].lt(backtest_start)].dropna(subset=["qty"])
if train_rows.empty:
    raise ValueError("No training rows precede the selected holdout period.")

lgbm_model = LGBMRegressor(
    objective="regression_l1",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=40,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)
lgbm_model.fit(train_rows[feature_columns], train_rows["qty"], categorical_feature=categorical_features)

pre_holdout = analysis_monthly.loc[analysis_monthly["Month"].lt(backtest_start)].copy()
known_series = pre_holdout[series_keys].drop_duplicates()
actual_holdout_all = analysis_monthly.loc[analysis_monthly["Month"].isin(backtest_months)].copy()
actual_holdout = actual_holdout_all.merge(known_series, on=series_keys, how="inner", validate="many_to_one")
new_series_holdout_rows = len(actual_holdout_all) - len(actual_holdout)
naive_predictions = recursive_quantity_forecast(pre_holdout, backtest_months, model=None)
lgbm_predictions = recursive_quantity_forecast(pre_holdout, backtest_months, model=lgbm_model)
backtest_results = actual_holdout[series_keys + ["Territory", "Month", "qty"]].rename(columns={"qty": "actual_qty"})
backtest_results = backtest_results.merge(
    naive_predictions[series_keys + ["Month", "qty"]].rename(columns={"qty": "seasonal_naive_qty"}),
    on=series_keys + ["Month"], how="left", validate="one_to_one"
)
backtest_results = backtest_results.merge(
    lgbm_predictions[series_keys + ["Month", "qty"]].rename(columns={"qty": "lightgbm_qty"}),
    on=series_keys + ["Month"], how="left", validate="one_to_one"
)
if backtest_results[["seasonal_naive_qty", "lightgbm_qty"]].isna().any().any():
    raise ValueError("Backtest predictions are missing for known series-month rows.")

def forecast_score(frame, prediction_column, dimensions=None):
    dimensions = dimensions or []
    if dimensions:
        scored = frame.groupby(dimensions + ["Month"], dropna=False, as_index=False).agg(
            actual=("actual_qty", "sum"), forecast=(prediction_column, "sum")
        )
    else:
        scored = frame[["Month", "actual_qty", prediction_column]].rename(
            columns={prediction_column: "forecast", "actual_qty": "actual"}
        )
    errors = scored["forecast"] - scored["actual"]
    denominator = scored["actual"].abs().sum()
    return {
        "MAE": errors.abs().mean(),
        "WAPE_pct": 100 * errors.abs().sum() / denominator if denominator else np.nan,
        "Bias": errors.sum(),
    }

score_rows = []
for model_name, prediction_column in [
    ("Seasonal naive", "seasonal_naive_qty"),
    ("LightGBM", "lightgbm_qty"),
]:
    score_rows.append({"model": model_name, "grain": "series", **forecast_score(backtest_results, prediction_column)})
    for dimension in ["Customer", "Product Family", "Territory", "Company"]:
        score_rows.append({
            "model": model_name,
            "grain": dimension,
            **forecast_score(backtest_results, prediction_column, [dimension]),
        })
backtest_metrics = pd.DataFrame(score_rows)
print(f"Holdout scored {len(known_series):,} existing series; {new_series_holdout_rows:,} rows for series first appearing inside the holdout were excluded.")
display(backtest_metrics.pivot(index="grain", columns="model", values=["MAE", "WAPE_pct", "Bias"]).round(3))

series_scores = backtest_metrics.loc[backtest_metrics["grain"].eq("series")].set_index("model")
lightgbm_wins = bool(series_scores.loc["LightGBM", "WAPE_pct"] < series_scores.loc["Seasonal naive", "WAPE_pct"])
print("Model selection by series-level WAPE:", "LightGBM beats seasonal naive." if lightgbm_wins else "LightGBM does not beat seasonal naive.")

Holdout scored 4,805 existing series; 1,618 rows for series first appearing inside the holdout were excluded.


MAE                WAPE_pct                 \
model             LightGBM Seasonal naive LightGBM Seasonal naive   
grain                                                               
Company         397987.991     567712.083   15.034         21.445   
Customer          1396.157       1686.022   50.907         61.476   
Product Family  124095.060     158623.963   21.094         26.963   
Territory        31716.670      36315.425   23.962         27.436   
series             666.618        815.414   60.400         73.882   

                       Bias                 
model              LightGBM Seasonal naive  
grain                                       
Company         6806878.701     12794110.0  
Customer        6806878.701     12794110.0  
Product Family  6806878.701     12794110.0  
Territory       6806878.701     12794110.0  
series          6806878.701     12794110.0

Model selection by series-level WAPE: LightGBM beats seasonal naive.


## Percentage summary

WAPE is a volume-weighted error percentage; lower is better. The 100-minus-WAPE column is only an informal accuracy-like complement. Review results at both detailed series and aggregate levels.

In [4]:
# Percentage summary for the current holdout.
wape_by_grain = backtest_metrics.pivot(index="grain", columns="model", values="WAPE_pct")
percentage_report = pd.DataFrame({
    "grain": wape_by_grain.index,
    "lightgbm_wape_pct": wape_by_grain["LightGBM"].to_numpy(),
    "lightgbm_100_minus_wape_pct": (100 - wape_by_grain["LightGBM"]).to_numpy(),
    "seasonal_naive_wape_pct": wape_by_grain["Seasonal naive"].to_numpy(),
    "wape_reduction_vs_naive_pct": (
        (wape_by_grain["Seasonal naive"] - wape_by_grain["LightGBM"])
        / wape_by_grain["Seasonal naive"].replace(0, np.nan) * 100
    ).to_numpy(),
})
print("WAPE is error (lower is better). 100 minus WAPE is an informal complement, not formal accuracy.")
display(percentage_report.round(2))

WAPE is error (lower is better). 100 minus WAPE is an informal complement, not formal accuracy.


,grain,lightgbm_wape_pct,lightgbm_100_minus_wape_pct,seasonal_naive_wape_pct,wape_reduction_vs_naive_pct
0,Company,15.03,84.97,21.45,29.90
1,Customer,50.91,49.09,61.48,17.19
2,Product Family,21.09,78.91,26.96,21.77
3,Territory,23.96,76.04,27.44,12.66
4,series,60.40,39.60,73.88,18.25
